# Webcam (WebGazer): browser recording to BIDS derivatives

This notebook runs the complete Pyxations workflow on a jsPsych/WebGazer
export. Install the optional detector and animation dependencies with
`pip install "pyxations[remodnav,video]"` before running it.

In [ ]:
import base64
from pathlib import Path

from IPython.display import HTML

repo = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)

In [ ]:
import pyxations as pyx

## Dataset to BIDS

The example contains a WebGazer CSV export. Pyxations interprets the first underscore-separated filename token as the source subject ID and the next token as the session label.

Where "arx" is the subject id and "first" is the session name, both separated by an underscore.

In [ ]:
output_folder = repo / "generated"
files_folder_path = repo / "examples" / "webgazer_antisaccade"
bids_dataset_folder = pyx.dataset_to_bids(
    output_folder,
    files_folder_path,
    "antisacadas_dataset",
    format_name="webgazer",
    overwrite=True,
)

In [ ]:
print(bids_dataset_folder)

## Compute derivatives

In [ ]:
dataset_type = "webgazer"
detection_algorithm = "remodnav"
pyx.compute_derivatives_for_dataset(
    bids_dataset_folder,
    dataset_type,
    detection_algorithm,
    overwrite=True,
    screen_height=768,
    screen_width=1024,
)

## Animating the gaze trace

`SampleVisualization` animates the sample stream directly, without
relying on detected events. The animation below covers the first 300
samples, about one minute of this recording, since one frame is rendered
per sample.

In [ ]:
experiment = pyx.Experiment(bids_dataset_folder)
experiment.load_data(detection_algorithm)
samples = experiment.get_session("0001", "antisacadas").samples().head(300)

animation_path = repo / "generated" / "webgazer_trace.gif"
pyx.SampleVisualization(samples, screen_width=1024, screen_height=768).animate(
    display=False, out_file=animation_path
)

# Embed as a data URI: the HTML export renders html outputs but falls back
# to a text repr for image/gif ones.
encoded = base64.b64encode(animation_path.read_bytes()).decode()
HTML(f'<img src="data:image/gif;base64,{encoded}" alt="gaze trace">')